# Дополнительное задание к ЛР2: автоэнкодер для аномалий SMAP

Отдельный файл версии 2 (`lab02_TensorFlow_v2`). Полный путь классификатора — в `lab2_classification_nn.ipynb`.

Автоэнкодер учится **только на нормальной** телеметрии `SMAP_train` и ищет сбои по ошибке восстановления. Так проще ловить **новые** типы аномалий, которых не было в разметке классификатора.

**Роль:** Лукавенко — архитектура, обучение, порог, оценка.


## Теория: автоэнкодер

## 1. Теория

### 1.1. Какую задачу решаем
Спутник постоянно передаёт на Землю **телеметрию** — показания датчиков и отметки о поданных командах. Почти всё время подсистемы работают штатно, но иногда в показаниях появляется **аномалия**: сбой, отказ, необычный режим. Человек не может круглосуточно просматривать сотни тысяч чисел, поэтому нужна модель, которая по куску телеметрии сама скажет, норма это или неисправность.

Ответ здесь — один из двух классов: **0 — норма**, **1 — неисправность**. Это **бинарная классификация**. Если бы классов было больше (норма, перегрев, сбой питания…), задача была бы **многоклассовой**.

Классификация отличается от **регрессии** видом ответа. В регрессии ответ — число на непрерывной шкале (как остаточный ресурс батареи в ЛР1), в классификации — метка класса. 

### 1.2. Обучение с учителем
Модели показывают много примеров, у которых **известен правильный ответ**, и она подбирает свои внутренние числа так, чтобы отвечать правильно.

* **Признаки (X)** — то, что подаём на вход: числа, описывающие кусок телеметрии.
* **Метка (y)** — правильный ответ: 0 или 1.
* **Модель** — формула с настраиваемыми числами (**весами**), которая превращает X в ответ.

Данные делят на три части:
* **train** — на нём модель учится;
* **validation** — на нём следят за обучением и выбирают настройки (когда остановиться, какой порог), но не учатся;
* **test** — «экзамен», на который смотрят один раз в конце. По нему судят о качестве.

### 1.3. Из временного ряда — в таблицу
Телеметрия — **временной ряд**: измерения идут одно за другим. Полносвязная сеть работает с **таблицей**: строка — пример, столбец — признак. Чтобы перейти от ряда к таблице, ряд режут на **окна** — короткие куски по 50 отсчётов — и описывают каждое окно несколькими числами: средним, разбросом, минимумом, максимумом, изменением за окно, средним скачком. Одно окно — одна строка таблицы.

Зачем окна, а не отдельные строки: одно мгновение почти ничего не говорит, а неисправность видна в **поведении** сигнала — рост, дрожание, провал.

**Метка окна** — 1, если внутри есть хотя бы один аномальный отсчёт. Окна идут с шагом 10, поэтому соседние окна перекрываются на 80% и почти одинаковы. Это важно для разбиения (1.10).

### 1.4. Нейрон и полносвязный слой
**Нейрон** берёт входные числа, умножает каждое на свой вес, складывает, добавляет смещение $b$ и пропускает результат через **функцию активации** $f$:

$$ y = f\big(w_1 x_1 + w_2 x_2 + \dots + w_n x_n + b\big) $$

**Слой Dense (полносвязный)** — набор нейронов, где каждый нейрон смотрит на **все** входы. Если на входе $n_{in}$ чисел, а в слое $n_{out}$ нейронов, у слоя $n_{in} \cdot n_{out} + n_{out}$ параметров: веса плюс по одному смещению на нейрон.

Сеть — несколько слоёв подряд. Первый слой находит простые сочетания признаков, следующие — сочетания этих сочетаний. Аналогия: жюри из 64 экспертов смотрит на все показатели окна, каждый со своими приоритетами; следующее жюри смотрит уже на мнения первого и т. д., последний судья выносит вердикт.

### 1.5. Функции активации

| Активация | Формула | Где используем | Смысл |
|---|---|---|---|
| **ReLU** | $\max(0, z)$ | скрытые слои | отрицательное обнуляет, положительное пропускает |
| **Sigmoid** | $\dfrac{1}{1 + e^{-z}}$ | выход бинарной классификации | превращает любое число в вероятность от 0 до 1 |
| **Softmax** | $\dfrac{e^{z_k}}{\sum_j e^{z_j}}$ | выход многоклассовой задачи | делит вероятность между классами, сумма = 1 |

Без нелинейной активации сколько слоёв ни складывай, получится одна линейная формула — та же логистическая регрессия. ReLU даёт сети «изгибы», и она может описывать сложные границы между нормой и сбоем.

Классов два, поэтому на выходе **один** нейрон с сигмоидой: он выдаёт $p$ — вероятность неисправности, вероятность нормы равна $1 - p$.

### 1.6. Функция потерь: бинарная кросс-энтропия
Функция потерь — число, которое показывает, насколько модель ошибается. Обучение — это поиск весов, при которых она минимальна. Для двух классов используют **бинарную кросс-энтропию**:

$$ L = -\frac{1}{m}\sum_{i=1}^{m}\Big[y_i \ln p_i + (1 - y_i)\ln(1 - p_i)\Big] $$

Если окно аномальное ($y = 1$), а сеть уверенно сказала $p = 0{,}01$, штраф $-\ln 0{,}01 \approx 4{,}6$. Если сказала $p = 0{,}9$, штраф всего $\approx 0{,}1$. Кросс-энтропия особенно сильно наказывает **уверенные ошибки**.

### 1.7. Как сеть учится
* **Градиентный спуск.** Считаем, в какую сторону надо сдвинуть каждый вес, чтобы ошибка уменьшилась (это **градиент**), и делаем небольшой шаг в эту сторону. Как спуск с горы вслепую: ощупываем уклон и шагаем вниз.
* **Обратное распространение ошибки** — способ быстро посчитать градиенты для всех слоёв: ошибка на выходе «растекается» от последнего слоя к первому.
* **Adam** — улучшенный градиентный спуск: сам подбирает размер шага для каждого веса.
* **Батч** — порция примеров (у нас 32 окна), после которой делается один шаг обновления весов.
* **Эпоха** — один полный проход по всем обучающим примерам.

### 1.8. Переобучение, Dropout и ранняя остановка
**Переобучение** — сеть запоминает обучающие примеры вместе с их шумом и хуже работает на новых. На графиках это видно так: ошибка на train продолжает падать, а на validation начинает расти.

* **Dropout(0.3)** на каждом шаге обучения случайно выключает 30% нейронов слоя. Сеть не может полагаться на отдельных «любимчиков» и учит устойчивые закономерности. При предсказании Dropout отключён — работает вся сеть.
* **EarlyStopping** следит за ошибкой на validation. Если она несколько эпох подряд не улучшается, обучение останавливается, и сеть возвращает веса лучшей эпохи.

### 1.9. Стандартизация и утечка данных
Признаки разного масштаба: флаги команд — 0 или 1, сигнал — от −1 до 1, разброс — сотые доли. Сеть чувствительна к масштабу, поэтому каждый признак приводят к среднему 0 и разбросу 1:

$$ z = \frac{x - \mu}{\sigma} $$

**Утечка данных** — когда в обучение просачивается информация из теста. Тогда метрики выглядят лучше, чем модель есть на самом деле. Поэтому $\mu$ и $\sigma$ считают **только по train**, а validation и test пересчитывают той же меркой.

### 1.10. Разбиение для временных рядов
Перемешивать окна случайно нельзя: соседние окна почти одинаковы, и у экзаменационного окна почти всегда найдётся «близнец» в обучении. Модель сдаст экзамен, просто вспомнив ответ.

Поэтому ряд режут на **блоки** (у нас по 5 000 отсчётов), и каждый блок целиком попадает в одну часть. На краях блока оставляют зазор, чтобы окна соседних блоков не делили ни одной точки. `StratifiedGroupKFold` раскладывает блоки так, чтобы доля аномалий в частях была похожей: *Group* — блоки не делятся, *Stratified* — доли классов выровнены.

### 1.11. Дисбаланс классов и SMOTE
Аномальных окон около 13%. Если ничего не делать, сеть быстро поймёт, что выгодно всегда отвечать «норма»: accuracy будет высокой, а найдено не будет ничего. Способы борьбы:
* **веса классов** (`class_weight`): ошибка на редком классе штрафуется сильнее;
* **пересэмплирование**: размножить редкий класс или проредить частый;
* **SMOTE** — создать новые искусственные примеры редкого класса;
* подбор **порога** и метрики, честные к редкому классу.

**SMOTE** берёт аномальное окно $x$, одного из его ближайших аномальных соседей $x_{nn}$ и ставит новую точку на отрезке между ними:

$$ x_{new} = x + \lambda\,(x_{nn} - x), \qquad \lambda \in [0, 1] $$

SMOTE применяют **только к train**. На test модель должна проверяться на настоящих данных с настоящей долей аномалий.

### 1.12. Baseline
**Baseline** — простая модель, с которой сравнивают сеть. Мы берём **логистическую регрессию**: это один нейрон с сигмоидой и без скрытых слоёв, граница между классами у неё прямая. Если сеть не лучше неё, скрытые слои ничего не дали. Ещё один ориентир — ответ «всегда норма»: он показывает, сколько accuracy можно получить, вообще ничего не находя.

### 1.13. Метрики классификации
Любой ответ попадает в одну из четырёх клеток **матрицы ошибок**. Положительный класс — неисправность:

|  | Модель: «неисправность» | Модель: «норма» |
|---|---|---|
| **На деле неисправность** | **TP** — нашли | **FN** — пропустили |
| **На деле норма** | **FP** — ложная тревога | **TN** — верно «норма» |

* **Accuracy** $= \dfrac{TP + TN}{TP + TN + FP + FN}$ — доля верных ответов. При 87% нормы ответ «всегда норма» даёт 0,87 и при этом бесполезен.
* **Precision** $= \dfrac{TP}{TP + FP}$ — из всех тревог сколько настоящих. Низкая precision — много ложных тревог.
* **Recall** $= \dfrac{TP}{TP + FN}$ — из всех неисправностей сколько нашли. Низкий recall — пропускаем настоящие сбои; для спутника это обычно опаснее.
* **F1** $= \dfrac{2 \cdot P \cdot R}{P + R}$ — одно число, компромисс precision и recall.
* **ROC-AUC** — площадь под ROC-кривой (доля найденных сбоев против доли ложных тревог при всех порогах). Это вероятность того, что случайное аномальное окно получит больший балл, чем случайное нормальное: 0,5 — угадывание, 1 — идеал.
* **PR-AUC** — площадь под кривой precision–recall. Для редкого класса она строже ROC-AUC: её уровень случайного угадывания равен доле аномалий (около 0,13), а не 0,5.

### 1.14. Порог решения
Сеть выдаёт вероятность, а решение «неисправность / норма» принимают по **порогу**: обычно 0,5. Ниже порог — больше найденных сбоев (recall растёт), но и больше ложных тревог (precision падает). Порог можно подбирать, но **только на validation**: подбор на test — это подгонка под экзамен.

### 1.15. Оценка по участкам
Аномалия обычно длится не одно окно, а целый **участок**. Для оператора важно, заметила ли модель сам случай сбоя, а не каждое его окно. Участок считаем **найденным**, если тревога сработала хотя бы в одном его окне.

### 1.16. Автоэнкодер: поиск аномалий без меток
**Автоэнкодер** (кодер-декодер) — сеть, которая учится выдавать на выходе то же, что получила на входе. Посередине у неё «узкое горлышко»:

```
окно 1250 чисел → кодер → 16 чисел → декодер → восстановленное окно 1250 чисел
```

**Кодер** сжимает окно до 16 чисел, **декодер** по ним восстанавливает окно обратно. Через узкое горлышко нельзя протащить всё подряд, поэтому сеть запоминает только типичное поведение сигнала.

Обучаем автоэнкодер **только на нормальных данных**. Он хорошо восстанавливает то, что похоже на норму, и плохо — то, чего никогда не видел. **Ошибка восстановления** и есть балл аномальности:

$$ e = \max_t \big(x_t - \hat x_t\big)^2 $$

где $x_t$ — исходный сигнал в момент $t$, $\hat x_t$ — восстановленный. Если ошибка выше порога, окно считается аномальным.

Главное отличие от классификатора: **метки при обучении не нужны**. Порог тоже выбирается без меток — по нормальным данным: например, так, чтобы 95% нормальных окон оказались ниже него. Метки нужны только для того, чтобы проверить качество.


## Подготовка данных

Загрузка SMAP, окна и разбиение блоками — как в основном ноутбуке.


In [ ]:
# Импорты и настройки запуска
import warnings
warnings.filterwarnings("ignore")   # не засоряем вывод служебными предупреждениями

import importlib                     # загрузка библиотеки по имени (для запасного чтения TsFile)
import random                        # стандартный генератор случайных чисел Python
from pathlib import Path             # удобная работа с путями к файлам

import numpy as np                   # массивы и математика
import pandas as pd                  # таблицы
import matplotlib.pyplot as plt      # графики
import seaborn as sns                # графики поверх matplotlib (countplot, heatmap)

from sklearn.model_selection import StratifiedGroupKFold   # разбиение целыми группами с похожей долей классов
from sklearn.preprocessing import StandardScaler           # приведение признаков к среднему 0 и разбросу 1
from sklearn.linear_model import LogisticRegression        # простая модель для сравнения
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, classification_report,
                             roc_auc_score, roc_curve, average_precision_score,
                             precision_recall_curve)       # метрики качества классификации
from imblearn.over_sampling import SMOTE                   # досоздание примеров редкого класса

import tensorflow as tf                           # библиотека нейросетей
import keras                                      # Keras: сборка и обучение сетей поверх TensorFlow
from keras.models import Sequential               # сеть как список слоёв
from keras.layers import Input, Dense, Dropout    # вход, полносвязный слой, выключение нейронов
from keras.callbacks import EarlyStopping         # ранняя остановка обучения

plt.rcParams["figure.figsize"] = (10, 4)   # размер графиков по умолчанию
pd.set_option("display.width", 160)        # таблицы печатаются в одну строку, без переносов

SEED = 42                                  # одно зерно случайности на всю работу
random.seed(SEED)                          # фиксируем случайность Python
np.random.seed(SEED)                       # фиксируем случайность NumPy
keras.utils.set_random_seed(SEED)          # фиксируем случайность TensorFlow (и заодно Python и NumPy)

DATA_DIR = Path("data")   # папка с данными рядом с ноутбуком
WINDOW = 50               # длина окна, отсчётов
STRIDE = 10               # шаг окна: следующее окно начинается на 10 отсчётов позже
BLOCK = 5000              # длина блока для разбиения, отсчётов
GAP = WINDOW              # зазор на краях блока: окна соседних блоков не пересекаются

print("TensorFlow", tf.__version__)   # проверка: какая версия TensorFlow загрузилась

In [ ]:
# Толмачева. Загрузка SMAP: из кэша или из файлов TsFile
def load_smap():
    cache = DATA_DIR / "cache" / "smap.npz"            # быстрый кэш с уже прочитанными данными
    if cache.exists():                                  # если кэш есть — берём из него
        z = np.load(cache, allow_pickle=True)
        return z["Xtr"], z["ytr"], z["Xte"], z["yte"], [str(c) for c in z["names"]]
    to_dataframe = importlib.import_module("tsfile").to_dataframe   # иначе читаем файлы TsFile (нужен pip install tsfile)
    out = []
    for name in ("SMAP_train.tsfile", "SMAP_test.tsfile"):
        df = to_dataframe(str(DATA_DIR / "raw" / name))                                    # файл -> таблица
        df = df.drop(columns=[c for c in df.columns if str(c).lower() in ("time", "timestamp")])  # время не признак
        label = [c for c in df.columns if str(c).lower().split(".")[-1] == "label"][0]     # столбец с меткой
        feats = df.drop(columns=[label]).select_dtypes("number")                           # 25 каналов
        out.append((feats.to_numpy(np.float32), df[label].to_numpy(int), list(feats.columns)))
    (Xtr, ytr, names), (Xte, yte, _) = out
    return Xtr, ytr, Xte, yte, names


Xtr_raw, ytr_raw, Xte_raw, yte_raw, CH_NAMES = load_smap()   # train: только норма; test: с метками 0/1
df = pd.DataFrame(Xte_raw, columns=CH_NAMES)                  # таблица из файла test: 25 каналов
df["fault"] = yte_raw                                         # целевой столбец: 0 — норма, 1 — неисправность

print(f"Файл train: {Xtr_raw.shape}, меток: {np.unique(ytr_raw)}")   # в train только метка 0
print(f"Размер данных (файл test): {df.shape}")                     # строки x (25 каналов + метка)
print(df.head())                                                    # первые 5 строк
print(f"\nРаспределение целевой переменной:\n{df['fault'].value_counts()}")   # сколько нормы и сбоев

In [ ]:
# Бурова. Окна по 50 отсчётов, 6 статистик на канал, метка окна
STATS = ["mean", "std", "min", "max", "trend", "jump"]          # какие статистики считаем
FEAT_NAMES = [f"{s}_{c}" for s in STATS for c in CH_NAMES]      # имена 150 признаков, например mean_channel_0


def window_features(Xb, yb):
    st = np.arange(0, len(Xb) - WINDOW + 1, STRIDE)             # начала окон: 0, 10, 20, ...
    idx = st[:, None] + np.arange(WINDOW)[None, :]              # номера строк каждого окна, форма (окна, 50)
    w = Xb[idx]                                                 # сами окна, форма (окна, 50, 25 каналов)
    jump = np.abs(np.diff(w, axis=1)).mean(1)                   # средний модуль скачка между соседними отсчётами
    F = np.concatenate([w.mean(1), w.std(1), w.min(1), w.max(1), w[:, -1] - w[:, 0], jump], axis=1)   # 6 x 25 = 150
    return F.astype(np.float32), yb[idx].max(1), st             # признаки, метка окна (1, если есть сбой), начала


feats, labels, groups, win_start = [], [], [], []
for g, a in enumerate(range(0, len(X_rows), BLOCK)):           # идём по ряду блоками по 5000 строк, g — номер блока
    b = min(a + BLOCK, len(X_rows))                            # конец блока (последний может быть короче)
    a2, b2 = a + GAP, b - GAP                                  # отступаем по 50 строк от краёв блока
    if b2 - a2 < WINDOW:                                       # слишком короткий хвост — пропускаем
        continue
    F, L, st = window_features(X_rows[a2:b2], y_rows[a2:b2])  # окна только внутри блока
    feats.append(F); labels.append(L); groups.append(np.full(len(L), g)); win_start.append(st + a2)

X = np.concatenate(feats)               # все окна: (число окон, 150)
y = np.concatenate(labels)              # метка каждого окна
groups = np.concatenate(groups)         # номер блока каждого окна — понадобится для разбиения
win_start = np.concatenate(win_start)   # где в файле начинается окно — для разбора ошибок
print(f"Окон: {len(y)}, признаков: {X.shape[1]}, блоков: {len(np.unique(groups))}")
print(f"Доля аномальных окон: {y.mean():.3f}")

In [ ]:
# Монахова. Разбиение целыми блоками, доли аномалий в частях похожи
def group_split(seed):
    sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=seed)   # 5 частей, блоки не делятся
    folds = [te for _, te in sgkf.split(X, y, groups)]                          # номера окон каждой части
    test_idx, val_idx = folds[0], folds[1]                                      # 1 часть — test, 1 — validation
    train_idx = np.setdiff1d(np.arange(len(y)), np.r_[test_idx, val_idx])       # остальные 3 части — train
    return train_idx, val_idx, test_idx


train_idx, val_idx, test_idx = group_split(SEED)
assert not set(groups[train_idx]) & set(groups[test_idx])   # проверка: ни один блок не попал и в train, и в test
assert not set(groups[train_idx]) & set(groups[val_idx])    # ... и в train, и в validation
assert not set(groups[val_idx]) & set(groups[test_idx])     # ... и в validation, и в test

for name, idx in (("Train", train_idx), ("Val", val_idx), ("Test", test_idx)):
    print(f"{name:5s}: окон {len(idx):6d} ({len(idx) / len(y):.0%}), блоков {len(np.unique(groups[idx])):3d}, "
          f"аномальных окон {y[idx].mean():.1%}")

In [ ]:
# Индексы для оценки автоэнкодера
y_test = y[test_idx]
n_blocks = int(groups.max()) + 1
b1, b2 = int(0.6 * n_blocks), int(0.8 * n_blocks)
chrono = (
    np.where(groups < b1)[0],                         # первые 60% блоков
    np.where((groups >= b1) & (groups < b2))[0],      # следующие 20%
    np.where(groups >= b2)[0],                        # последние 20% — «новые» подсистемы
)
print(f"test: {len(test_idx)} окон, новые подсистемы: {len(chrono[2])} окон")


---
## Детектор аномалий: автоэнкодер (кодер-декодер)
**Ответственная: Лукавенко**

Классификатор из разделов 6–10 учится на размеченных сбоях и узнаёт сбои знакомых подсистем, а на новых подсистемах не работает (раздел 10.2). Здесь другой подход: **учим кодер-декодер только на нормальной телеметрии**, а потом прогоняем через него все данные. Нормальный сигнал он воспроизводит хорошо, а на сбое ошибается — по величине ошибки восстановления и находим неисправность.

Для обучения берём файл `SMAP_train`: в нём только норма тех же подсистем. С файлом test он не пересекается, поэтому утечки нет. Метки файла test используем только для проверки качества.

### 1. Данные для автоэнкодера

Автоэнкодер восстанавливает сам сигнал окна: 50 отсчётов × 25 каналов = 1 250 чисел. Из каждого канала вычитаем его значение в начале окна. Так у разных подсистем, которые держат сигнал на разном уровне, остаётся только **форма** сигнала внутри окна.

*Сначала пробовали* восстанавливать окно как есть, с уровнем, и брать за балл среднюю ошибку по окну. Тогда балл больше зависел от того, насколько шумная подсистема, чем от сбоя. Решили убрать уровень и брать **максимальную** ошибку по сигналу `channel_0`: так ловятся короткие провалы и скачки.


In [ ]:
# Лукавенко. Окна нормальной телеметрии из SMAP_train для автоэнкодера
def raw_windows(X_src, starts):
    idx = starts[:, None] + np.arange(WINDOW)[None, :]   # номера строк каждого окна
    w = X_src[idx]                                       # окна: (число окон, 50 отсчётов, 25 каналов)
    w = w - w[:, :1, :]                                  # убираем уровень: каждый канал окна начинается с 0
    return w.reshape(len(w), -1).astype(np.float32)      # 50 x 25 = 1250 чисел на окно

ae_starts = np.arange(0, len(Xtr_raw) - WINDOW + 1, STRIDE)   # окна по всему файлу SMAP_train (только норма)
ae_cut = int(len(ae_starts) * 0.8)                            # первые 80% окон — обучение, последние 20% — проверка
A_fit = raw_windows(Xtr_raw, ae_starts[:ae_cut])              # нормальные окна для обучения
A_val = raw_windows(Xtr_raw, ae_starts[ae_cut:])              # нормальные окна для ранней остановки и порога
A_all = raw_windows(X_rows, win_start)                        # все окна файла test в том же порядке, что X и y

print(f"Обучение: {A_fit.shape}, проверка на норме: {A_val.shape}, окна файла test: {A_all.shape}")


### 2. Архитектура кодер-декодер

```
1250 → Dense 256 → Dense 64 → Dense 16 (код) → Dense 64 → Dense 256 → Dense 1250
       └──────── кодер ────────┘               └──────── декодер ────────┘
```

Кодер постепенно сжимает окно до 16 чисел, декодер зеркально разворачивает обратно. В скрытых слоях ReLU, на выходе линейная активация: восстанавливаем числа любого знака. Функция потерь — **MSE**, средний квадрат разницы между входом и выходом: сеть учится, чтобы выход был как можно ближе ко входу.


In [ ]:
# Лукавенко. Автоэнкодер: кодер сжимает окно до 16 чисел, декодер восстанавливает
def build_autoencoder(n_in):
    model = Sequential([
        Input(shape=(n_in,)),             # вход: окно 1250 чисел
        Dense(256, activation="relu"),    # кодер: 1250 -> 256
        Dense(64, activation="relu"),     # кодер: 256 -> 64
        Dense(16, activation="relu"),     # код — «узкое горлышко»: окно описано 16 числами
        Dense(64, activation="relu"),     # декодер: 16 -> 64
        Dense(256, activation="relu"),    # декодер: 64 -> 256
        Dense(n_in),                      # выход: восстановленное окно 1250 чисел, линейная активация
    ])
    model.compile(optimizer="adam", loss="mse")   # учим выход повторять вход
    return model


keras.utils.set_random_seed(SEED)                 # повторяемая случайность
autoencoder = build_autoencoder(A_fit.shape[1])
autoencoder.summary()


### 3. Обучение только на норме

Вход и правильный ответ — одно и то же окно: `fit(A_fit, A_fit)`. Ранняя остановка следит за ошибкой восстановления на нормальных окнах проверки.


In [ ]:
# Лукавенко. Обучение автоэнкодера: на входе и на выходе одно и то же нормальное окно
ae_history = autoencoder.fit(
    A_fit, A_fit,                                   # учимся восстанавливать вход
    validation_data=(A_val, A_val),                 # проверка на нормальных окнах
    epochs=60,                                      # не больше 60 проходов
    batch_size=64,                                  # шаг после каждых 64 окон
    callbacks=[EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True)],
    verbose=0,
)
ae_best = int(np.argmin(ae_history.history["val_loss"])) + 1
print(f"Эпох: {len(ae_history.history['loss'])}, лучшая по val_loss: {ae_best}, "
      f"ошибка восстановления на норме: {min(ae_history.history['val_loss']):.5f}")

plt.plot(ae_history.history["loss"], label="train (норма)")
plt.plot(ae_history.history["val_loss"], label="validation (норма)")
plt.title("Автоэнкодер: ошибка восстановления по эпохам"); plt.xlabel("Эпоха"); plt.ylabel("MSE")
plt.legend(); plt.grid(True, alpha=0.3); plt.show()


### 4. Балл аномальности и порог

**Балл окна** — самая большая ошибка восстановления сигнала `channel_0` внутри окна. **Порог** считаем по нормальным окнам проверки из `SMAP_train`: порог 95% значит, что 95% нормальных окон получили балл ниже него. Метки файла test для выбора порога не используются.


In [ ]:
# Лукавенко. Балл аномальности = максимальная ошибка восстановления сигнала, порог — по норме
def anomaly_score(model, A):
    R = model.predict(A, batch_size=2048, verbose=0)        # восстановленные окна
    err = ((R - A) ** 2).reshape(len(A), WINDOW, -1)        # квадрат ошибки для каждого отсчёта и канала
    return err[:, :, 0].max(1)                              # балл окна: худший отсчёт сигнала channel_0


score_val = anomaly_score(autoencoder, A_val)               # баллы нормальных окон проверки
score_all = anomaly_score(autoencoder, A_all)               # баллы всех окон файла test
QUANTS = (0.80, 0.90, 0.95, 0.99)                           # пороги: какая доля нормы окажется ниже
ae_thr = {q: float(np.quantile(score_val, q)) for q in QUANTS}
AE_Q = 0.95                                                 # основной порог
AE_THR = ae_thr[AE_Q]
print("Пороги по нормальным окнам:", {f"{int(q * 100)}%": round(t, 4) for q, t in ae_thr.items()})

fig, ax = plt.subplots(1, 3, figsize=(17, 4))
s_te = score_all[test_idx]
bins = np.logspace(np.log10(max(s_te.min(), 1e-6)), np.log10(s_te.max()), 60)
ax[0].hist(s_te[y_test == 0], bins=bins, alpha=0.6, log=True, label="норма")             # число окон, ось Y логарифмическая
ax[0].hist(s_te[y_test == 1], bins=bins, alpha=0.6, log=True, label="неисправность")
ax[0].axvline(AE_THR, c="k", ls=":", label="порог 95%")
ax[0].set_xscale("log"); ax[0].set_title("Баллы окон test"); ax[0].set_xlabel("ошибка восстановления")
ax[0].set_ylabel("число окон"); ax[0].legend()

anom = test_idx[y_test == 1]; norm = test_idx[y_test == 0]
i_bad = anom[np.argmax(score_all[anom])]                                     # аномальное окно с самым большим баллом
i_ok = norm[np.argsort(score_all[norm])[len(norm) // 2]]                     # типичное нормальное окно
for a, i, title in ((ax[1], i_ok, "Нормальное окно"), (ax[2], i_bad, "Аномальное окно")):
    rec = autoencoder.predict(A_all[i:i + 1], verbose=0).reshape(WINDOW, -1)[:, 0]
    a.plot(A_all[i].reshape(WINDOW, -1)[:, 0], label="исходный сигнал")
    a.plot(rec, "--", label="восстановленный")
    a.set_title(f"{title}: балл {score_all[i]:.3g}"); a.set_xlabel("отсчёт в окне"); a.legend()
plt.tight_layout(); plt.show()


### 5. Качество: какие сбои найдены и сколько ложных тревог

Главная мера — **участки**: участок сбоя найден, если тревога сработала хотя бы в одном его окне. Рядом — доля ложных тревог на нормальных окнах. Проверяем на тех же тестовых блоках, что и классификатор, и на **новых подсистемах** — последних 20% блоков, где классификатор не работал (раздел 10.2).

Для честного сравнения порог классификатора выбираем так же: по его нормальным окнам validation, на тех же долях 80–99%.


In [ ]:
# Лукавенко. Найденные участки и ложные тревоги автоэнкодера
def detect_events(score, thr, idx):
    found = total = 0
    for g in np.unique(groups[idx]):
        m = idx[groups[idx] == g]
        order = np.argsort(win_start[m])
        yt, alarm = y[m][order], score[m][order] > thr
        d = np.diff(np.r_[0, yt, 0])
        for a, b in zip(np.where(d == 1)[0], np.where(d == -1)[0]):
            total += 1
            found += int(alarm[a:b].any())
    false_alarm = 100 * (score[idx][y[idx] == 0] > thr).mean()
    return found, total, false_alarm


new_idx = chrono[2]
rows = {}
for q in QUANTS:
    f1_, t1_, fa1 = detect_events(score_all, ae_thr[q], test_idx)
    f2_, t2_, fa2 = detect_events(score_all, ae_thr[q], new_idx)
    rows[f"порог {int(q * 100)}%"] = {
        "test: участков": f"{f1_}/{t1_}", "test: ложн. тревог %": round(fa1, 1),
        "новые: участков": f"{f2_}/{t2_}", "новые: ложн. тревог %": round(fa2, 1),
    }
ae_table = pd.DataFrame(rows).T
print(ae_table.to_string())

pred_ae = (score_all[test_idx] > AE_THR).astype(int)
print(f"\nПо окнам test, порог 95%: precision {precision_score(y_test, pred_ae, zero_division=0):.3f}, "
      f"recall {recall_score(y_test, pred_ae):.3f}, ROC-AUC {roc_auc_score(y_test, score_all[test_idx]):.3f}")
print("Сравнение с Dense-классификатором — в lab2_classification_nn.ipynb, раздел 11.5.")


In [ ]:
# Лукавенко. Балл автоэнкодера вдоль времени на одном тестовом блоке с аномалией
blk = [g for g in np.unique(groups[test_idx]) if y[groups == g].any()]
g0 = max(blk, key=lambda g: y[groups == g].sum())          # блок, где аномальных окон больше всего
m0 = np.where(groups == g0)[0]
m0 = m0[np.argsort(win_start[m0])]

fig, ax = plt.subplots(figsize=(14, 3.5))
ax.plot(win_start[m0], score_all[m0], lw=0.8, label="балл автоэнкодера")
ax.fill_between(win_start[m0], 0, score_all[m0].max(), where=y[m0] == 1, color="tab:red", alpha=0.2, label="истинная аномалия")
ax.axhline(AE_THR, c="k", ls=":", label="порог 95%")
ax.set_yscale("log"); ax.set_xlabel("номер отсчёта в файле test"); ax.set_ylabel("ошибка восстановления")
ax.set_title(f"Блок {g0}: где автоэнкодер поднимает тревогу"); ax.legend(loc="upper right"); plt.show()


### 6. Вывод по автоэнкодеру

Автоэнкодер учился **только на нормальной телеметрии**, без единой метки, и порог тоже выбран по норме. При пороге 95% он находит 9 участков сбоя из 16 на тестовых блоках, поднимая тревогу на 1,7% нормальных окон.

**На знакомых подсистемах классификатор сильнее.** При той же доле ложных тревог (1,7%) он находит 12 участков из 16. Это ожидаемо: классификатор учился на размеченных сбоях именно этих подсистем.

**На новых подсистемах автоэнкодер работает, а классификатор нет.** Там классификатор хуже монетки (ROC-AUC 0,38, раздел 10.2), а автоэнкодер при пороге 95% находит 10 участков из 22 при 2,1% ложных тревог, при пороге 90% — 14 из 22 при 6,6%. Он выучил, как выглядит нормальный сигнал, а не как выглядят сбои конкретных подсистем.

**По отдельным окнам автоэнкодер слабый:** ROC-AUC 0,38. Многие аномалии в SMAP длинные и спокойные, и такой сигнал восстанавливается легко. Автоэнкодер ловит не весь аномальный участок, а момент, когда сигнал резко ломается, — и этого хватает, чтобы участок был найден.

**Порог — компромисс** между пропусками и ложными тревогами. При 80% находится больше участков, но ложных тревог 16–18%. При 99% ложных тревог почти нет, но найдено меньше трети участков.

Обучение остановилось на лимите в 60 эпох: ошибка восстановления на норме к этому моменту ещё немного снижалась.
